# Cell Cycle Marker Intensity Quantification

Segments nuclei from DAPI fluorescence images and quantifies Ki67 cell cycle marker intensity per cell alongside IRF3 nuclear localization. A per-nucleus Spearman correlation between DAPI and Ki67 intensities is computed. Results are exported to CSV for downstream statistical analysis.

## 1. Dependencies

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

## 2. Configuration

In [ ]:
# --- Input ---
directory = "image4_npy"  # Input folder; each .npy file is a (3, H, W) array: [IRF3, DAPI, Ki67]

# --- Group mapping ---
group_map = {
    '17': 'Mock_32H',
    '18': 'GS_32H'
}
order = ['17', '18']

# --- Nucleus segmentation parameters ---
global_blur_size = (55, 55)     # Gaussian blur size for DAPI before thresholding
specific_blur_size = (65, 65)   # Blur within each ROI to refine nuclear mask
min_area = 3500  # Minimum valid nucleus area (pixels)
circularity_cutoff = 0.65  # Shape filter to exclude elongated or fragmented objects (established range: 0.6–0.7)
rescue_threshold = -20  # Percent-difference cutoff for classification of nuclear IRF3 localization

# Morphology ring radii (~pixels)
inner_dilate = 11  # Inner erosion radius; excludes the 10 px immediately adjacent to the nuclear boundary
outer_dilate = 31  # Outer dilation radius; net cytoplasmic ring is ~20 px (outer 30 px minus excluded inner 10 px)

## 3. Image Processing Pipeline

In [ ]:
# ===============================================================
# MAIN LOOP
# ===============================================================
delta_by_group = {k: [] for k in group_map}
results_by_group = {k: [] for k in group_map}
all_areas = []
circularities = []

for filename in sorted(os.listdir(directory)):
    if not filename.endswith(".npy"):
        continue

    prefix = filename.split("_")[0]
    if prefix not in group_map:
        continue

    condition = group_map[prefix]
    path = os.path.join(directory, filename)
    data = np.load(path)

    if data.shape[0] < 3:
        print(f"[WARN] {filename}: missing channels, skipping")
        continue

    irf3_gray = data[0].astype(np.uint8)
    dapi_gray = data[1].astype(np.uint8)
    ki_gray = data[2].astype(np.uint8)
    height, width = dapi_gray.shape

    # --- Segment nuclei from DAPI ---
    blurred = cv2.GaussianBlur(dapi_gray, global_blur_size, 0)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    areas = [cv2.contourArea(cnt) for cnt in contours]
    all_areas.extend(areas)

    def is_inside(cnt):
        x, y, w, h = cv2.boundingRect(cnt)
        return x > 0 and y > 0 and (x + w) < width and (y + h) < height

    filtered_contours = [
        cnt for cnt in contours
        if cv2.contourArea(cnt) >= min_area and is_inside(cnt)
    ]

    for j, cnt in enumerate(filtered_contours):
        x, y, w, h = cv2.boundingRect(cnt)
        pad_x = int(w * 0.25)
        pad_y = int(h * 0.25)

        x1 = max(x - pad_x, 0)
        y1 = max(y - pad_y, 0)
        x2 = min(x + w + pad_x, width)
        y2 = min(y + h + pad_y, height)

        roi_irf3 = irf3_gray[y1:y2, x1:x2]
        roi_dapi = dapi_gray[y1:y2, x1:x2]
        roi_ki67 = ki_gray[y1:y2, x1:x2]

        roi_blur = cv2.GaussianBlur(roi_dapi, specific_blur_size, 0)
        _, roi_thresh = cv2.threshold(roi_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        roi_contours, _ = cv2.findContours(roi_thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if not roi_contours:
            continue

        roi_contours = sorted(roi_contours, key=cv2.contourArea, reverse=True)
        largest_cnt = roi_contours[0]

        area = cv2.contourArea(largest_cnt)
        perimeter = cv2.arcLength(largest_cnt, True)
        if perimeter == 0:
            continue

        circularity = 4 * np.pi * (area / (perimeter ** 2))
        circularities.append(circularity)
        if area < min_area or circularity < circularity_cutoff:
            continue

        # --- Create masks ---
        mask_inside = np.zeros(roi_dapi.shape, dtype=np.uint8)
        cv2.drawContours(mask_inside, [largest_cnt], -1, 255, -1)

        kernel_inner = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (inner_dilate, inner_dilate))
        kernel_outer = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (outer_dilate, outer_dilate))
        dilated_inner = cv2.dilate(mask_inside, kernel_inner, iterations=1)
        dilated_outer = cv2.dilate(mask_inside, kernel_outer, iterations=1)
        ring_mask = cv2.subtract(dilated_outer, dilated_inner)

        background_mask = (roi_irf3 > 10).astype(np.uint8) * 255
        cytoring_mask = cv2.bitwise_and(ring_mask, background_mask)

        # --- IRF3 intensities ---
        mean_inside = cv2.mean(roi_irf3, mask=mask_inside)[0]
        mean_cytoring = cv2.mean(roi_irf3, mask=cytoring_mask)[0]
        if mean_cytoring == 0:
            continue
        mean_dapi = cv2.mean(roi_dapi, mask=mask_inside)[0]
        total_dapi = float(cv2.sumElems(cv2.bitwise_and(roi_dapi, roi_dapi, mask=mask_inside))[0])

        delta = mean_inside - mean_cytoring
        percent_diff = 100 * delta / mean_cytoring

        # --- ki67 density ---
        mask = np.zeros(roi_ki67.shape, dtype=np.uint8)
        cv2.drawContours(mask, [largest_cnt], -1, 255, -1)
        area = np.count_nonzero(mask)
        density_ki67 = np.sum(roi_ki67[mask == 255]) / area if area > 0 else np.nan

        # --- Spearman correlation (DAPI vs ki67) ---
        def normalize_to_uint8(img):
            img = img.astype(np.float32)
            if img.max() == img.min():
                return np.zeros_like(img, dtype=np.uint8)
            img = 255 * (img - img.min()) / (img.max() - img.min())
            return img.astype(np.uint8)

        roi_dapi_norm = normalize_to_uint8(roi_dapi)
        roi_ki67_norm = normalize_to_uint8(roi_ki67)
        dapi_vals = roi_dapi_norm[mask == 255].ravel()
        ki_vals = roi_ki67_norm[mask == 255].ravel()

        if len(dapi_vals) >= 2:
            spearman_r, _ = spearmanr(ki_vals, dapi_vals)
        else:
            spearman_r = np.nan

        classification = "nuclear" if percent_diff >= rescue_threshold else "non-nuclear"

        # --- Save result ---
        results_by_group[prefix].append({
            "label": f"{filename}_cell{j}",
            "condition": condition,
            "percent_diff": percent_diff,
            "classification": classification,
            "ki67_density": density_ki67,
            "spearman_corr": spearman_r,
            "irf3_mean": mean_inside,
            "dapi_mean": mean_dapi,
            "dapi_total": total_dapi,
            "area": area
        })
        delta_by_group[prefix].append(delta)

## 4. Export Results

In [ ]:
# ===============================================================
# Save Data
# ===============================================================
df = pd.concat([pd.DataFrame(v) for v in results_by_group.values()], ignore_index=True)
df.to_csv("image4_results.csv", index=False)